## Практична

## завдання 1

In [2]:
from IPython.display import display, HTML

# Створення таблиці за допомогою чистого HTML
html_table = """
<table style="width:100%; border-collapse: collapse; font-family: sans-serif; text-align: left;">
    <thead>
        <tr style="background-color: #f2f2f2; border-bottom: 2px solid #ddd;">
            <th style="padding: 12px; font-weight: bold;">Архітектура</th>
            <th style="padding: 12px; font-weight: bold;">Кількість шарів</th>
            <th style="padding: 12px; font-weight: bold;">Особливості</th>
            <th style="padding: 12px; font-weight: bold;">Застосування</th>
        </tr>
    </thead>
    <tbody>
        <tr style="border-bottom: 1px solid #ddd;">
            <td style="padding: 12px; font-weight: bold; vertical-align: top;">LeNet-5</td>
            <td style="padding: 12px; vertical-align: top;">5 шарів з параметрами (2 згорткові + 3 повнозв'язні)</td>
            <td style="padding: 12px; vertical-align: top;">Використовує активації Sigmoid/Tanh та Average Pooling. Історично перша успішна згорткова мережа.</td>
            <td style="padding: 12px; vertical-align: top;">Розпізнавання рукописних символів (цифр на чеках, поштових індексів). Набір даних MNIST.</td>
        </tr>
        <tr style="border-bottom: 1px solid #ddd;">
            <td style="padding: 12px; font-weight: bold; vertical-align: top;">AlexNet</td>
            <td style="padding: 12px; vertical-align: top;">8 шарів (5 згорткових + 3 повнозв'язні)</td>
            <td style="padding: 12px; vertical-align: top;">Першою використала функцію ReLU, шар Dropout для запобігання перенавчанню, Max Pooling та GPU для прискорення навчання.</td>
            <td style="padding: 12px; vertical-align: top;">Класифікація великих баз зображень (переможець ImageNet 2012). Початок сучасного буму Deep Learning.</td>
        </tr>
        <tr style="border-bottom: 1px solid #ddd;">
            <td style="padding: 12px; font-weight: bold; vertical-align: top;">ResNet</td>
            <td style="padding: 12px; vertical-align: top;">50, 101 або 152 шари (надглибока мережа)</td>
            <td style="padding: 12px; vertical-align: top;">Впровадила залишкові зв'язки (Skip/Residual Connections). Вони дозволяють градієнту проходити крізь сотні шарів без затухання.</td>
            <td style="padding: 12px; vertical-align: top;">Сучасний стандарт-бекбон для комп'ютерного зору: класифікація, детекція об'єктів, сегментація зображень.</td>
        </tr>
    </tbody>
</table>
"""

# Відображення таблиці в інтерфейсі Jupyter
display(HTML(html_table))

Архітектура,Кількість шарів,Особливості,Застосування
LeNet-5,5 шарів з параметрами (2 згорткові + 3 повнозв'язні),Використовує активації Sigmoid/Tanh та Average Pooling. Історично перша успішна згорткова мережа.,"Розпізнавання рукописних символів (цифр на чеках, поштових індексів). Набір даних MNIST."
AlexNet,8 шарів (5 згорткових + 3 повнозв'язні),"Першою використала функцію ReLU, шар Dropout для запобігання перенавчанню, Max Pooling та GPU для прискорення навчання.",Класифікація великих баз зображень (переможець ImageNet 2012). Початок сучасного буму Deep Learning.
ResNet,"50, 101 або 152 шари (надглибока мережа)",Впровадила залишкові зв'язки (Skip/Residual Connections). Вони дозволяють градієнту проходити крізь сотні шарів без затухання.,"Сучасний стандарт-бекбон для комп'ютерного зору: класифікація, детекція об'єктів, сегментація зображень."


## завдання 2

In [4]:
import torch
import torchvision
import torchvision.transforms as transforms

# Визначення трансформації (перетворення у тензор та нормалізація)
# Використання стандартних коефіцієнтів для датасету CIFAR-10
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.4914, 0.4822, 0.4465), 
                         std=(0.2023, 0.1994, 0.2010))
])

train_dataset = torchvision.datasets.CIFAR10(
    root='./data', 
    train=True, 
    download=True, 
    transform=transform
)

# Створення DataLoader
batch_size = 64
train_loader = torch.utils.data.DataLoader(
    train_dataset, 
    batch_size=batch_size, 
    shuffle=True
)

# Отримання та перевірка структури одного батча
data_iter = iter(train_loader)
images, labels = next(data_iter)

# Виведення результатів
print("=== СТРУКТУРА ОДНОГО БАТЧА ===")
print(f"Форма тензора зображень (images shape): {images.shape}")
print(f"Форма тензора міток (labels shape): {labels.shape}")
print(f"Тип даних зображень (images dtype): {images.dtype}")
print(f"Тип даних міток (labels dtype): {labels.dtype}")
print(f"Розмір поточного батча: {len(images)} зображень")

100%|██████████| 170M/170M [18:25<00:00, 154kB/s]  


=== СТРУКТУРА ОДНОГО БАТЧА ===
Форма тензора зображень (images shape): torch.Size([64, 3, 32, 32])
Форма тензора міток (labels shape): torch.Size([64])
Тип даних зображень (images dtype): torch.float32
Тип даних міток (labels dtype): torch.int64
Розмір поточного батча: 64 зображень


## завдання 3

In [10]:
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models

# Перевірка наявності GPU для прискорення навчання
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Використовується пристрій: {device}")

# =====================================================================
# ПОБУДОВА МЕРЕЖІ LENET-5
# =====================================================================
class LeNet5(nn.Module):
    def __init__(self):
        super(LeNet5, self).__init__()
        # Вхід: 3 канали (RGB), вихід: 6 карт ознак. Згортка 5x5
        self.conv1 = nn.Conv2d(3, 6, kernel_size=5)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Вхід: 6, вихід: 16. Згортка 5x5
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Повнозв'язні шари. Після двох пулінгів розмір карти: 16 * 5 * 5
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10) # 10 класів CIFAR-10
        
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.pool1(self.relu(self.conv1(x)))
        x = self.pool2(self.relu(self.conv2(x)))
        x = x.view(-1, 16 * 5 * 5) # Вирівнювання в один вектор
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# =====================================================================
# НАЛАШТУВАННЯ RESNET18 (TRANSFER LEARNING)
# =====================================================================
# Завантажуємо переднавчену модель
resnet18 = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Заморожуємо всі параметри (ваги базових шарів не будуть оновлюватися)
for param in resnet18.parameters():
    param.requires_grad = False

# Замінюємо останній шар (класифікатор) на новий під 10 класів
# Нові шари за замовчуванням мають requires_grad=True
num_ftrs = resnet18.fc.in_features
resnet18.fc = nn.Linear(num_ftrs, 10)

# =====================================================================
# ФУНКЦІЯ ДЛЯ ТРЕНУВАННЯ ТА ОЦІНКИ
# =====================================================================
def train_model(model, train_loader, epochs=5):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    # Оптимізуємо лише ті параметри, які потребують градієнта (актуально для ResNet)
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)
    
    print(f"\nПочаток навчання моделі {model.__class__.__name__}...")
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
        epoch_loss = running_loss / len(train_loader)
        epoch_acc = 100.0 * correct / total
        print(f"Епоха [{epoch+1}/{epochs}] - Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.2f}%")
        
    return epoch_acc, epoch_loss

# =====================================================================
# ЗАПУСК НАВЧАННЯ
# =====================================================================
lenet_acc, lenet_loss = train_model(LeNet5(), train_loader, epochs=5)
resnet_acc, resnet_loss = train_model(resnet18, train_loader, epochs=5)

Використовується пристрій: cpu

Початок навчання моделі LeNet5...
Епоха [1/5] - Loss: 1.5978, Accuracy: 41.37%
Епоха [2/5] - Loss: 1.2936, Accuracy: 53.93%
Епоха [3/5] - Loss: 1.1719, Accuracy: 58.80%
Епоха [4/5] - Loss: 1.0838, Accuracy: 61.65%
Епоха [5/5] - Loss: 1.0193, Accuracy: 64.11%

Початок навчання моделі ResNet...
Епоха [1/5] - Loss: 1.7217, Accuracy: 39.80%
Епоха [2/5] - Loss: 1.5993, Accuracy: 44.43%
Епоха [3/5] - Loss: 1.5780, Accuracy: 44.99%
Епоха [4/5] - Loss: 1.5681, Accuracy: 45.56%
Епоха [5/5] - Loss: 1.5606, Accuracy: 45.61%


## завдання 4

In [11]:
# =====================================================================
# ВІДОБРАЖЕННЯ РЕЗУЛЬТАТІВ У ВИГЛЯДІ HTML-ТАБЛИЦІ
# =====================================================================
html_results = f"""
<table style="width:70%; border-collapse: collapse; font-family: sans-serif; text-align: left;">
    <thead>
        <tr style="background-color: #f2f2f2; border-bottom: 2px solid #ddd;">
            <th style="padding: 12px; font-weight: bold;">Модель</th>
            <th style="padding: 12px; font-weight: bold;">Попередньо навчена</th>
            <th style="padding: 12px; font-weight: bold;">Accuracy</th>
            <th style="padding: 12px; font-weight: bold;">Loss</th>
        </tr>
    </thead>
    <tbody>
        <tr style="border-bottom: 1px solid #ddd;">
            <td style="padding: 12px; font-weight: bold;">LeNet</td>
            <td style="padding: 12px;">ні</td>
            <td style="padding: 12px; font-weight: bold; color: #337ab7;">{lenet_acc:.2f}%</td>
            <td style="padding: 12px;">{lenet_loss:.4f}</td>
        </tr>
        <tr style="border-bottom: 1px solid #ddd;">
            <td style="padding: 12px; font-weight: bold;">ResNet18</td>
            <td style="padding: 12px;">так</td>
            <td style="padding: 12px; font-weight: bold; color: #337ab7;">{resnet_acc:.2f}%</td>
            <td style="padding: 12px;">{resnet_loss:.4f}</td>
        </tr>
    </tbody>
</table>
"""

display(HTML(html_results))

Модель,Попередньо навчена,Accuracy,Loss
LeNet,ні,64.11%,1.0193
ResNet18,так,45.61%,1.5606
